## Load Dataset

In [ ]:
!unzip -q /content/coco_subset_100.zip

In [1]:
import json

In [2]:
with open('/home/dal696598/scratch/coco_subset_100/annotations/instances_train2017.json', 'r') as js:
    annot = json.load(js)

In [3]:
class_map = {cat['id']:cat['name'] for cat in annot['categories']}
classes = list(class_map.values())

In [4]:
from collections import defaultdict

In [5]:
images = {img['id']:img['file_name'] for img in annot['images']}
annotations = defaultdict(list)
for an in annot['annotations']:
    annotations[an['image_id']].append([an['bbox'], class_map[an['category_id']]])

## Load Mask R CNN model

In [7]:
from os import path

from PIL import Image

import numpy as np

import torch

from torchvision.tv_tensors import BoundingBoxes, Mask
from torchvision import transforms
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.image_list import ImageList


In [8]:
device = torch.device('cuda') if torch.cuda.is_available() else "cpu"

In [9]:
weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = maskrcnn_resnet50_fpn_v2(weights=weights)
model.to(device=device)
model.name = 'maskrcnn_resnet50_fpn_v2'

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth" to /home/dal696598/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth
100%|██████████| 177M/177M [00:00<00:00, 310MB/s]  


In [10]:
images_dir = "/home/dal696598/scratch/coco_subset_100/train2017"

## Detect bounding boxes and ROIs

In [11]:
max_boxes_to_draw = 300
iou_threshold = 0.3

In [12]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter

    return inter / union if union > 0 else 0

In [13]:
total_matched = 0
total_preds = 0

results = {}
model.eval()
with torch.no_grad():
    for img_id in images:
        img_path = path.join(images_dir, images[img_id])
        image = Image.open(img_path).convert('RGB')
        input_tensor = transforms.ToTensor()(image).to(device)
        input_tensor = input_tensor.unsqueeze(0)
        image_list = ImageList(input_tensor, [(image.height, image.width)])

        img, _ = model.transform([input_tensor.squeeze(0)])
        features = model.backbone(img.tensors)
        rois, _ = model.rpn(img, features)
        output, _ = model.roi_heads(features, rois, img.image_sizes)
        output = model.transform.postprocess(output, img.image_sizes, [(image.height, image.width)])
        output = output[0]
        keep = (output['scores'] >= 0.3)
        scores, idx = output['scores'][keep].sort(descending=True)
        idx = idx[:300]

        boxes = output['boxes'][keep][idx]
        labels = output['labels'][keep][idx]
        scores = scores[:max_boxes_to_draw]

        gt_boxes = []
        gt_classes = []

        for bbox, cls in annotations[img_id]:
            x, y, w, h = bbox
            gt_boxes.append([x, y, x + w, y + h])
            gt_classes.append(cls)

        pred_boxes = boxes.cpu().numpy()
        pred_scores = scores.cpu().numpy()
        pred_labels = labels.cpu().numpy()

        rows = []
        matched = 0
        used_gt = set()

        order = np.argsort(-pred_scores)
        for i in order:
            pred_box = pred_boxes[i]
            pred_class = pred_labels[i]
            score = pred_scores[i]

            best_iou = 0
            best_j = -1

            for j, gt_box in enumerate(gt_boxes):
                if j in used_gt:
                    continue

                iou = compute_iou(pred_box, gt_box)

                if iou > best_iou:
                    best_iou = iou
                    best_j = j
            if best_iou >= iou_threshold:
                gt_box_match = gt_boxes[best_j]
                gt_class_match = gt_classes[best_j]
                used_gt.add(best_j)
                matched += 1
            else:
                gt_box_match = None
                gt_class_match = None

            rows.append({
                'pred_box': pred_box.tolist(),
                'pred_class': int(pred_class),
                'pred_score': float(score),
                'gt_box': gt_box_match,
                'gt_class': gt_class_match,
                'iou': float(best_iou)
            })
        results[img_id] = {
            'image_tensor': input_tensor.squeeze(0).cpu(),
            'rois': rois[0].cpu(),
            'predictions' : rows
        }
        total_matched += matched
        total_preds += len(pred_boxes)
        break

match_percentage = total_matched / total_preds if total_preds > 0 else 0

print("Total matched:", total_matched)
print("Total predictions:", total_preds)
print("Overall Match %:", match_percentage)


Total matched: 4
Total predictions: 4
Overall Match %: 1.0


In [46]:
torch.save(results, 'results.pt')

## Upload detections to HuggingFace

In [2]:
from huggingface_hub import HfApi
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
token = os.environ['token']

In [5]:
api = HfApi()
api.upload_file(
    path_or_fileobj='/content/results.pt',
    path_in_repo='Bboxes_and_256D',
    repo_id='preetsojitra/Echo-VilD',
    repo_type="dataset",
    token=token,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/results.pt         :   0%|          |  957kB /  333MB            

CommitInfo(commit_url='https://huggingface.co/datasets/preetsojitra/Echo-VilD/commit/ea6564a717ee91fb0951ad974c8e438a00ae5b15', commit_message='Upload Bboxes_and_256D with huggingface_hub', commit_description='', oid='ea6564a717ee91fb0951ad974c8e438a00ae5b15', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/preetsojitra/Echo-VilD', endpoint='https://huggingface.co', repo_type='dataset', repo_id='preetsojitra/Echo-VilD'), pr_revision=None, pr_num=None)